In [1]:
# NOTE:
# This import assumes that the project root directory is in PYTHONPATH.
# It works in PyCharm's Jupyter environment by default.
# If running in standalone Jupyter Lab, you may need to adjust
# the working directory or manually append the project root to sys.path.
from data.dataProcess import *

In [2]:
# Data reading
adata = ad.read_h5ad('nanostring_cosmx_human_nsclc_reference.h5ad')

In [3]:
# Data normalize
adata.X = adata.layers['counts'].copy()
sc.pp.normalize_total(adata, target_sum=1e4)
sc.pp.log1p(adata)

In [4]:
train_data_1 = adata[adata.obs['batch'] == 'lung5_rep1'].copy()
test_data = adata[adata.obs['batch'] == 'lung5_rep2'].copy()

In [5]:
# Remove rare niches with very few cells.
cell_type_col = "niche"
train_data_1, info = filter_rare_cell_types_logspace(
    train_data_1,
    cell_type_col=cell_type_col,
    sigma_cut=0.1,
    eps=1e-8,
    verbose=True
)

log-space stats: mean=8.082, std=2.450
threshold (count): 2532
cell types to remove (3): ['neutrophils', 'immune', 'macrophages']
removing 1.89% of samples (1758/93206)


In [6]:
# To mimic real-world scenarios, all niches were retained during test set construction.
test_data.obs[cell_type_col].value_counts()

stroma                         22783
myeloid-enriched stroma        20998
lymphoid structure             16096
tumor interior                 15602
plasmablast-enriched stroma    10003
tumor-stroma boundary           9209
neutrophils                     2584
immune                           212
Name: niche, dtype: int64

In [7]:
train_data_1.X = train_data_1.X.toarray()
test_data.X = test_data.X.toarray()

In [8]:
type_list = list(train_data_1.obs[cell_type_col].unique())
type_list

['stroma',
 'plasmablast-enriched stroma',
 'lymphoid structure',
 'tumor-stroma boundary',
 'myeloid-enriched stroma',
 'tumor interior']

In [9]:
# Based on the size of the blank tissue section,
# chose the sliding-window width and overlap ratio, 
# and computed the corresponding summary statistics.
coords = train_data_1.obsm["spatial"]
x = coords[:, 0]
y = coords[:, 1]

x_min, x_max = x.min(), x.max()
y_min, y_max = y.min(), y.max()
width = x_max - x_min
res = spatial_sliding_window_stats(
    train_data_1,
    window_width=width / 250,
    overlap_rate=0.9
)

print("Total number of windows:", res["n_windows"])
print("Average cells per window:", res["mean_cells_per_window"])

Total number of windows: 2491
Average cells per window: 365.2862304295464


In [10]:
dp = data_process(type_list, 'nsclc', rand_n=3000, rand_cell_num=np.floor(res["mean_cells_per_window"]),
                  label_key=cell_type_col, spatial_key='spatial')

In [11]:
train_datas = [train_data_1]
test_datas = [test_data]

In [12]:
train_x_sim_list = []
train_x_latent_list = []
train_y_list = []

test_x_sim_list = []
test_x_latent_list = []
test_y_list = []

In [13]:
# By setting different rotation angles for the blank tissue section, we generated additional training samples.
angels = [0, 30, 45, 60, 90]

In [14]:
# Construct the training dataset based on the blank tissue section and the sliding window.
for trainData in train_datas:
    for angel in angels:
        x_sim, y = dp.generate_pseudo_bulk(trainData, angle_deg=angel,
                                           strip_width=width / 250, overlap_ratio=0.9, min_cells=res["mean_cells_per_window"]/3, min_cell_types=2)
        train_x_sim_list += x_sim
        train_y_list += y

Success rate: 100.0%
Success rate: 80.9%
Success rate: 79.9%
Success rate: 82.0%
Success rate: 100.0%


In [15]:
# Construct training dataset by randomly sampling the data.
train_x_sim_list_, train_y_list_ = dp.build_pseudo_bulk_no_noise(train_data_1)
train_x_sim_list += train_x_sim_list_
train_y_list += train_y_list_

100%|██████████| 3000/3000 [07:52<00:00,  6.35it/s]


In [16]:
# Generate the test dataset.
for testData in test_datas:
    for angel in angels:
        x_sim, y = dp.generate_pseudo_bulk(testData, angle_deg=angel,
                                           strip_width=width / 250, overlap_ratio=0, min_cells=res["mean_cells_per_window"]/3, min_cell_types=2)
        test_x_sim_list += x_sim
        test_y_list += y

Success rate: 100.0%
Success rate: 83.5%
Success rate: 80.4%
Success rate: 83.2%
Success rate: 100.0%


In [17]:
len(train_x_sim_list), len(test_x_sim_list)

(14984, 1215)

In [18]:
train = [train_x_sim_list, train_y_list]
test = [test_x_sim_list, test_y_list]
with open(f'{dp.tissue_name}_nonorm', 'wb') as f:
    pickle.dump(train, f)
    pickle.dump(test, f)

In [19]:
train_x_sim_list = dp.normalize(train_x_sim_list)
test_x_sim_list = dp.normalize(test_x_sim_list)

In [20]:
train = [train_x_sim_list, train_y_list]
test = [test_x_sim_list, test_y_list]
with open(f'{dp.tissue_name}_norm', 'wb') as f:
    pickle.dump(train, f)
    pickle.dump(test, f)

In [21]:
with open(f'{dp.tissue_name}_ref', 'wb') as f:
    pickle.dump(train_datas, f)